# Feature engineering for building permit cost prediction

This notebook transforms raw building permit data into a feature set suitable for
predicting `log_cost` (log-transformed construction cost). We load and clean the
data using the project's `data_loader` module, engineer new features, encode
categoricals, and examine correlations to guide feature selection.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

from src.data_loader import load_or_fetch_data, preprocess_data, engineer_features

pd.set_option('display.max_columns', 40)
sns.set_style('whitegrid')
%matplotlib inline

## 1. Load and preprocess the raw data

We use `load_or_fetch_data` to pull from the local cache (or the Calgary Open Data
API on first run), then pass the result through `preprocess_data` which handles
type conversions, removes rows with missing or non-positive costs, trims the top
and bottom 1% outliers, and extracts date components.

In [ ]:
raw_df = load_or_fetch_data(data_dir='../data', limit=500000)
print(f'Raw records: {len(raw_df):,}')
print(f'Columns: {list(raw_df.columns)}')

In [ ]:
df = preprocess_data(raw_df)
print(f'After preprocessing: {len(df):,} rows, {df.shape[1]} columns')
df.head(3)

## 2. Engineer core features

`engineer_features` creates `log_cost` (our target), `log_sqft`, `cost_per_sqft`,
and community-level aggregates (`community_avg_cost`, `community_median_cost`,
`community_permit_count`). These aggregates act as a form of target encoding
for the community dimension.

In [ ]:
df = engineer_features(df)
print(f'Columns after feature engineering: {df.shape[1]}')
print(df[['log_cost', 'log_sqft', 'community_avg_cost',
           'community_median_cost', 'community_permit_count']].describe().round(2))

## 3. Inspect feature distributions before and after transformations

The raw `estprojectcost` is heavily right-skewed. Taking `log1p` produces a
distribution much closer to normal, which helps linear models like Ridge and
also stabilises variance for tree-based methods.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(df['estprojectcost'].dropna(), bins=80, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Raw estprojectcost', fontsize=11)
axes[0, 0].set_xlabel('Cost ($)')

axes[0, 1].hist(df['log_cost'].dropna(), bins=80, edgecolor='black', alpha=0.7, color='teal')
axes[0, 1].set_title('log_cost (log1p transform)', fontsize=11)
axes[0, 1].set_xlabel('log(1 + cost)')

axes[1, 0].hist(df['totalsqft'].dropna(), bins=80, edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Raw totalsqft', fontsize=11)
axes[1, 0].set_xlabel('Square feet')

axes[1, 1].hist(df['log_sqft'].dropna(), bins=80, edgecolor='black', alpha=0.7, color='teal')
axes[1, 1].set_title('log_sqft (log1p transform)', fontsize=11)
axes[1, 1].set_xlabel('log(1 + sqft)')

plt.tight_layout()
plt.show()

## 4. Create interaction features

Permit class groups capture the type of work (residential, commercial, etc.).
Interacting `totalsqft` with permit class dummies lets the model learn that
a square foot of commercial construction costs differently from residential.

In [ ]:
# One-hot encode permitclassgroup for interaction terms
pcg_dummies = pd.get_dummies(df['permitclassgroup'], prefix='pcg', drop_first=True)
print(f'Permit class group dummies: {list(pcg_dummies.columns)}')

# Multiply each dummy by totalsqft to create interaction features
for col in pcg_dummies.columns:
    interaction_name = f'sqft_x_{col}'
    df[interaction_name] = df['totalsqft'].fillna(0) * pcg_dummies[col].values

interaction_cols = [c for c in df.columns if c.startswith('sqft_x_')]
print(f'\nCreated {len(interaction_cols)} interaction features')
df[interaction_cols].describe().round(1)

## 5. Target encoding for categorical variables

Instead of plain label encoding, we compute the mean `log_cost` per category.
This preserves ordinal information about how expensive each permit type or
work class tends to be. We apply a smoothing factor to prevent overfitting
for categories with few observations.

In [ ]:
def target_encode(df, col, target='log_cost', smoothing=10):
    """Smoothed target encoding: blend category mean with global mean."""
    global_mean = df[target].mean()
    agg = df.groupby(col)[target].agg(['mean', 'count'])
    smooth = (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)
    return df[col].map(smooth)

cat_cols = ['permittype', 'permitclassgroup', 'workclassgroup']
for col in cat_cols:
    if col in df.columns:
        encoded_name = f'{col}_te'
        df[encoded_name] = target_encode(df, col)
        print(f'{col} -> {encoded_name}  |  nunique={df[col].nunique()}, '
              f'encoded range=[{df[encoded_name].min():.2f}, {df[encoded_name].max():.2f}]')

## 6. Correlation matrix

We examine pairwise Pearson correlations among the numeric features and the
target to identify strong predictors and detect multicollinearity.

In [ ]:
numeric_features = [
    'log_cost', 'totalsqft', 'log_sqft', 'housingunits',
    'apply_year', 'apply_month', 'latitude', 'longitude',
    'community_avg_cost', 'community_median_cost', 'community_permit_count',
    'permittype_te', 'permitclassgroup_te', 'workclassgroup_te',
]
available_numeric = [c for c in numeric_features if c in df.columns]

corr_matrix = df[available_numeric].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, ax=ax, linewidths=0.5)
ax.set_title('Feature correlation matrix', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Correlations with target, sorted by absolute value
target_corr = corr_matrix['log_cost'].drop('log_cost').abs().sort_values(ascending=False)
print('Top correlations with log_cost:\n')
print(target_corr.to_string())

## 7. Feature selection rationale

Based on the analysis above, we select the following features for modeling:

| Feature | Rationale |
|---|---|
| `permittype_te` | Strong correlation with cost; target-encoded avoids high cardinality |
| `permitclassgroup_te` | Captures residential vs. commercial vs. industrial cost differences |
| `workclassgroup_te` | New construction vs. renovation vs. demolition cost profiles differ |
| `totalsqft` / `log_sqft` | Physical size is a primary cost driver |
| `housingunits` | Multi-unit projects cost more; useful for residential permits |
| `apply_year`, `apply_month` | Captures construction cost inflation and seasonal patterns |
| `latitude`, `longitude` | Spatial proxy for land values and neighbourhood characteristics |
| `community_avg_cost` | Community-level baseline cost (smoothed target encoding) |
| `sqft_x_pcg_*` | Interaction terms let models adjust per-sqft cost by permit class |

We drop `community_median_cost` because it is highly correlated with
`community_avg_cost` (r > 0.95), which would introduce multicollinearity
in Ridge regression without adding predictive power.

In [ ]:
# Assemble final feature set
from src.model import CATEGORICAL_FEATURES, NUMERICAL_FEATURES, TARGET

print(f'Categorical features (from model.py): {CATEGORICAL_FEATURES}')
print(f'Numerical features (from model.py):   {NUMERICAL_FEATURES}')
print(f'Target variable: {TARGET}')

In [ ]:
# Quick check: missing value summary for model features
all_features = CATEGORICAL_FEATURES + NUMERICAL_FEATURES
available = [c for c in all_features if c in df.columns]
missing_pct = df[available].isnull().mean().sort_values(ascending=False) * 100
print('Missing value percentage per feature:\n')
print(missing_pct[missing_pct > 0].to_string())
if missing_pct.sum() == 0:
    print('No missing values in selected features.')

## 8. Save engineered dataset

We persist the feature-engineered dataframe so downstream notebooks
(`03_modeling`, `04_evaluation`) can load it directly without repeating
these steps.

In [ ]:
import os
os.makedirs('../data', exist_ok=True)
df.to_csv('../data/building_permits_engineered.csv', index=False)
print(f'Saved engineered dataset: {len(df):,} rows, {df.shape[1]} columns')

## Summary

- Loaded 484K+ building permit records and cleaned them with `preprocess_data()`.
- Created `log_cost` and `log_sqft` transforms to reduce skewness.
- Built community-level aggregates as a proxy for neighbourhood effects.
- Generated `sqft * permit_class` interaction features.
- Applied smoothed target encoding to three categorical columns.
- Identified `community_avg_cost` and `permittype_te` as the strongest single predictors.
- Dropped `community_median_cost` due to high collinearity with `community_avg_cost`.

The next notebook (`03_modeling`) trains Ridge, Random Forest, Gradient Boosting,
and XGBoost models on this feature set.